# 04 — Data Processing

Process **today's** option-chain snapshots into tidy Parquet + daily metrics.

Run after Notebook 02. Only files for the current NSE session (`src/trading_calendar.py`: IST clock + NSE holidays, not the computer's date) are processed for the active symbols.

In [1]:
import os
import sys
import json
import glob
from datetime import datetime

import pandas as pd
import yaml

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
CONFIG_PATH = os.path.join(PROJECT_ROOT, "config", "config.yaml")
sys.path.insert(0, PROJECT_ROOT)

from src.chain_parser import load_chain_csv, compute_daily_metrics
from src.trading_calendar import resolve_session_date, describe_session

with open(CONFIG_PATH) as f:
    CONFIG = yaml.safe_load(f)

SESSION_OVERRIDE = None  # e.g. "2026-10-01" to reprocess a past session
TODAY = resolve_session_date(SESSION_OVERRIDE)
print(describe_session())
DATA_RAW = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED = os.path.join(PROJECT_ROOT, "data", "processed")
METRICS_DIR = os.path.join(DATA_PROCESSED, "daily_metrics")
CHAIN_DIR = os.path.join(DATA_PROCESSED, "option_chain")
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(CHAIN_DIR, exist_ok=True)

ACTIVE_SYMBOLS = CONFIG.get("collection", {}).get("active_symbols", ["NIFTY", "BANKNIFTY"])
print(f"Processing date: {TODAY} | Symbols: {ACTIVE_SYMBOLS}")


def load_meta(csv_path):
    meta_path = csv_path.replace(".csv", ".meta.json")
    if not os.path.exists(meta_path):
        return None
    with open(meta_path) as f:
        return json.load(f)


def chain_files_for_date(symbol, run_date):
    pattern = os.path.join(DATA_RAW, symbol.lower(), "option_chain", f"{run_date}_*.csv")
    paths = sorted(glob.glob(pattern))
    return [p for p in paths if "Select" not in os.path.basename(p)]


def process_chain_file(csv_path, meta):
    chain = load_chain_csv(csv_path)
    spot = meta.get("underlying_spot")
    metrics = compute_daily_metrics(chain, spot=spot)
    metrics.update({
        "symbol": meta.get("symbol", "").upper(),
        "date": meta.get("date", TODAY),
        "expiry": meta.get("expiry"),
        "asset_class": meta.get("asset_class"),
        "index_name": meta.get("index_name"),
        "expiry_type": meta.get("expiry_type"),
        "dte": meta.get("dte"),
        "expiry_regime": meta.get("expiry_regime"),
        "dte_bucket": meta.get("dte_bucket"),
        "is_front_expiry": meta.get("is_front_expiry"),
        "signal_regime": meta.get("signal_regime"),
        "source_file": csv_path,
    })

    tidy_path = csv_path.replace(DATA_RAW, CHAIN_DIR).replace(".csv", ".parquet")
    os.makedirs(os.path.dirname(tidy_path), exist_ok=True)
    chain.to_parquet(tidy_path, index=False)
    return metrics


def process_symbol(symbol, run_date=TODAY):
    paths = chain_files_for_date(symbol, run_date)
    if not paths:
        print(f"[SKIP] No chains for {symbol} on {run_date}")
        return pd.DataFrame()

    rows = []
    for path in paths:
        meta = load_meta(path)
        if meta is None:
            print(f"[SKIP] No metadata: {path}")
            continue
        try:
            rows.append(process_chain_file(path, meta))
            print(f"OK: {path}")
        except Exception as e:
            print(f"FAIL: {path} -> {e}")
    return pd.DataFrame(rows)


# ── RUN (session only, active symbols) ─────────────
_missing = [s for s in ACTIVE_SYMBOLS if not chain_files_for_date(s, TODAY)]
if _missing:
    raise FileNotFoundError(
        f"No raw chains for session {TODAY} for {_missing}. Run notebook 02 first "
        "(or set SESSION_OVERRIDE to reprocess an older session)."
    )

for sym in ACTIVE_SYMBOLS:
    df = process_symbol(sym, TODAY)
    if df.empty:
        continue
    out = os.path.join(METRICS_DIR, f"{sym.lower()}_{TODAY}.csv")
    df.to_csv(out, index=False)
    print(f"Metrics -> {out} ({len(df)} expiries)")
    display(df)


Session: 2026-10-07 (Wed) | now 2026-10-08 00:56 IST | 2026-10-08 is not a finished trading session
Processing date: 2026-10-07 | Symbols: ['NIFTY', 'BANKNIFTY']
OK: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_03-Nov-2026.csv
OK: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_09-Nov-2026.csv
OK: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_13-Oct-2026.csv
OK: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_19-Oct-2026.csv
OK: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_23-Nov-2026.csv
OK: d:\nse-options-scraper\data\raw\nifty\option_chain\2026-10-07_27-Oct-2026.csv
Metrics -> d:\nse-options-scraper\data\processed\daily_metrics\nifty_2026-10-07.csv (6 expiries)


,atm_strike,underlying_spot,pcr_oi,pcr_volume,total_call_oi,total_put_oi,oi_centroid_ce,oi_centroid_pe,composite_v0_chg_oi_atm_band,strike_count,...,expiry,asset_class,index_name,expiry_type,dte,expiry_regime,dte_bucket,is_front_expiry,signal_regime,source_file
0,22600.0,22603.05,0.468748,0.486496,16559.0,7762.0,23622.570807,22090.775573,440.0,104,...,03-Nov-2026,index,NIFTY,weekly,27,T-8+,11+,False,back_series,d:\nse-options-scraper\data\raw\nifty\option_c...
1,22600.0,22603.05,0.573913,0.676230,230.0,132.0,23821.521739,21975.000000,-23.0,71,...,09-Nov-2026,index,NIFTY,weekly,33,T-8+,11+,False,back_series,d:\nse-options-scraper\data\raw\nifty\option_c...
2,22600.0,22603.05,0.813680,1.052455,2110276.0,1717090.0,23269.745664,21816.276287,302820.0,108,...,13-Oct-2026,index,NIFTY,weekly,6,T-6,5-10,True,front_month,d:\nse-options-scraper\data\raw\nifty\option_c...
3,22600.0,22603.05,0.809837,0.896941,203878.0,165108.0,23366.404909,21727.138903,4313.0,104,...,19-Oct-2026,index,NIFTY,weekly,12,T-8+,11+,False,back_series,d:\nse-options-scraper\data\raw\nifty\option_c...
4,22600.0,22603.05,0.864686,0.870212,195434.0,168989.0,23873.418392,22627.734645,-6017.0,122,...,23-Nov-2026,index,NIFTY,weekly,47,T-8+,11+,False,back_series,d:\nse-options-scraper\data\raw\nifty\option_c...
5,22600.0,22603.05,0.973850,0.721643,1069269.0,1041308.0,23606.067369,22966.925252,29286.0,144,...,27-Oct-2026,index,NIFTY,weekly,20,T-8+,11+,False,back_series,d:\nse-options-scraper\data\raw\nifty\option_c...


OK: d:\nse-options-scraper\data\raw\banknifty\option_chain\2026-10-07_23-Nov-2026.csv
OK: d:\nse-options-scraper\data\raw\banknifty\option_chain\2026-10-07_27-Oct-2026.csv
Metrics -> d:\nse-options-scraper\data\processed\daily_metrics\banknifty_2026-10-07.csv (2 expiries)


,atm_strike,underlying_spot,pcr_oi,pcr_volume,total_call_oi,total_put_oi,oi_centroid_ce,oi_centroid_pe,composite_v0_chg_oi_atm_band,strike_count,...,expiry,asset_class,index_name,expiry_type,dte,expiry_regime,dte_bucket,is_front_expiry,signal_regime,source_file
0,55100.0,55055.55,0.615103,0.893297,85784.0,52766.0,57203.677842,55449.310162,-513.0,149,...,23-Nov-2026,index,BANKNIFTY,monthly,47,T-8+,11+,False,back_series,d:\nse-options-scraper\data\raw\banknifty\opti...
1,55100.0,55055.55,0.923828,0.958082,483853.0,446997.0,57311.127967,54353.867923,-7072.0,157,...,27-Oct-2026,index,BANKNIFTY,monthly,20,T-8+,11+,True,front_month,d:\nse-options-scraper\data\raw\banknifty\opti...
